# Kenya Malaria Incidence Analysis
## 1. Introduction

Malaria remains a major public health concern in Kenya,
affecting communities differently across the country.

Understanding how malaria cases vary across counties and
over time can help identify areas experiencing a higher
burden of malaria.

This project analyses malaria data in Kenya to identify
trends, compare malaria incidence across counties, and
highlight patterns that can support better public health
planning and decision-making.

The analysis uses Python for data cleaning, exploration,
analysis, and visualization.

## 2. Problem Statement

Malaria continues to affect many communities in Kenya,
but its distribution and burden are not the same across
all counties.

Without proper analysis of malaria data, it can be
difficult to identify counties with higher malaria
incidence, understand changes over time, and recognize
areas that may require greater attention.

This project addresses this problem by analysing malaria
data to identify geographical differences and trends
that can support informed public health decisions.

## 3. Project Objectives

### Main Objective

To analyse malaria cases in Kenya and identify geographical
and temporal patterns that can support public health
planning and decision-making.

### Specific Objectives

1. To examine the distribution of malaria cases across
   different counties in Kenya.
2. To identify counties with the highest and lowest
   average malaria incidence.
3. To analyse changes in malaria incidence over time.
4. To identify patterns and trends in malaria incidence.
5. To provide recommendations based on the findings.

## 4. Research Questions

1. How does malaria incidence vary across counties in Kenya?
2. Which counties record the highest and lowest average
   malaria incidence?
3. How has malaria incidence changed over the years?
4. Are there noticeable patterns in malaria incidence
   across the study period?
5. What recommendations can be made from the findings?



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('../data/malaria_kenya_original.xlsx')
df['date'] = pd.to_datetime(df['date'])
df['month_name'] = df['date'].dt.month_name()
df.head()

## 1. Data quality

In [ ]:
print('Shape:', df.shape)
print('Counties:', df['county'].nunique())
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Missing cells:', df.isna().sum().sum())
print('Duplicate county-year-month keys:', df.duplicated(['county','year','month']).sum())
print(df.groupby('county').size().describe())
print(df.groupby('year').size())

The provided file has 6,110 records, 47 counties and 130 records per county. The 2015 records begin in March; January and February are absent for all counties. 

## 2. Annual trend

In [ ]:
annual = df.groupby('year')['incidence_per_1000'].agg(['mean','median','count']).reset_index()
annual
plt.figure(figsize=(9,4))
plt.plot(annual['year'], annual['mean'], marker='o')
plt.title('Average malaria incidence by year')
plt.xlabel('Year'); plt.ylabel('Incidence per 1,000')
plt.grid(alpha=.25); plt.show()

2015 is partial and should be flagged. The 2016-2025 comparison is more appropriate for a full-year trend comparison.

## 3. County comparison

In [ ]:
county = df.groupby('county')['incidence_per_1000'].mean().sort_values(ascending=False)
print(county.head(10))
print(county.tail(10))
county.sort_values().tail(15).plot(kind='barh', figsize=(8,6), title='15 counties with highest average incidence')
plt.xlabel('Incidence per 1,000'); plt.show()

## 4. Monthly seasonality

In [ ]:
monthly = df.groupby('month')['incidence_per_1000'].mean()
monthly.index = [pd.Timestamp(2000,m,1).strftime('%b') for m in monthly.index]
monthly.plot(marker='o', figsize=(9,4), title='Average incidence by calendar month')
plt.ylabel('Incidence per 1,000'); plt.grid(alpha=.25); plt.show()

## 5. Environmental associations

In [ ]:
env = ['temperature','precipitation','NDVI','ITN_coverage']
print(df[['incidence_per_1000'] + env].corr()['incidence_per_1000'].sort_values(ascending=False))
fig, axes = plt.subplots(2,2,figsize=(9,7))
for ax, col in zip(axes.flat, env):
    ax.scatter(df[col], df['incidence_per_1000'], alpha=.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel('Incidence per 1,000')
plt.tight_layout(); plt.show()

Pooled correlations can be confounded by county differences, seasonality and time trends. They do not prove that any environmental factor causes malaria changes. For a stronger extension, demean variables within county and/or fit a panel model with county and month/year controls.

In [ ]:
vars_ = ['incidence_per_1000'] + env
within = df[vars_] - df.groupby('county')[vars_].transform('mean')
within.corr()['incidence_per_1000']

## 6. Transmission-state caution

In [ ]:
df.groupby('transmission_state')['incidence_per_1000'].agg(['count','mean','min','max'])

The publisher describes transmission_state as derived from incidence thresholds (0=no/low, 1=moderate, 2=high, 3=epidemic). It is a label constructed from the outcome, not an independent predictor. Obtain the data dictionary for exact thresholds before using it.